# 📊 SoukIQ (WafR) Advanced Business Intelligence Notebook
**Engagement:** Enterprise Business Analytics & Root Cause Analysis  
**Audience:** Chief Executive Officer (CEO) & Chief Operating Officer (COO)  

In this notebook, we execute advanced statistical and behavioral analytics directly from our PostgreSQL warehouse:
1. **Pareto (80/20) Analysis:** Quantifying merchant revenue concentration.
2. **RFM Customer Segmentation:** Clustering Hanouts based on Recency, Frequency, and Monetary scores.
3. **Cohort Retention Heatmap:** Tracking MoM customer lifecycle retention.
4. **Operations Friction Correlation:** Proving the statistical link between API outages and merchant churn.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2
import warnings

warnings.filterwarnings('ignore')

# Set professional consulting-style plot theme (BCG/Deloitte Palette)
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10

# Database Connection Details
DB_PARAMS = {
    "host": "localhost",
    "database": "soukiq_db",
    "user": "postgres",
    "password": "root", # ⚠️ نفس الباسورد ديال Ingestion
    "port": "5432"
}

def get_connection():
    return psycopg2.connect(**DB_PARAMS)

print("🔌 Connected to 'soukiq_db' successfully. Visual themes initialized.")

In [ ]:
# 1. Extract merchant revenue rankings
query_pareto = """
WITH MerchantRevenue AS (
    SELECT 
        merchant_id,
        SUM(amount_mad * commission_rate) AS merchant_net_revenue
    FROM fact_transaction
    WHERE status = 'Success'
    GROUP BY merchant_id
)
SELECT 
    merchant_id,
    merchant_net_revenue,
    ROW_NUMBER() OVER (ORDER BY merchant_net_revenue DESC) AS merchant_rank,
    SUM(merchant_net_revenue) OVER (ORDER BY merchant_net_revenue DESC) / SUM(merchant_net_revenue) OVER () * 100 AS cumulative_rev_pct
FROM MerchantRevenue
ORDER BY merchant_rank;
"""

conn = get_connection()
df_pareto = pd.read_sql(query_pareto, conn)
conn.close()

# Calculate total counts
total_merchants = len(df_pareto)
df_pareto['cumulative_merchant_pct'] = (df_pareto['merchant_rank'] / total_merchants) * 100

# Plotting the Pareto Curve (Corporate Navy & Teal)
fig, ax1 = plt.subplots(figsize=(11, 6))

# Bar Chart for individual merchant net revenue
ax1.bar(df_pareto['merchant_rank'][:100], df_pareto['merchant_net_revenue'][:100], color='#1f77b4', alpha=0.6, label='Net Revenue per Merchant (Top 100)')
ax1.set_xlabel('Merchant Rank (Ordered by Net Revenue)', fontsize=12)
ax1.set_ylabel('Net Revenue (MAD)', color='#1f77b4', fontsize=12)
ax1.tick_params(axis='y', labelcolor='#1f77b4')

# Line Chart for Cumulative Percentage
ax2 = ax1.twinx()
ax2.plot(df_pareto['merchant_rank'], df_pareto['cumulative_rev_pct'], color='#2ca02c', linewidth=2.5, label='Cumulative Revenue %')
ax2.set_ylabel('Cumulative Net Revenue (%)', color='#2ca02c', fontsize=12)
ax2.tick_params(axis='y', labelcolor='#2ca02c')

# Identify 80% mark
p_index = df_pareto[df_pareto['cumulative_rev_pct'] >= 80.0].iloc[0]
ax2.axhline(y=80, color='red', linestyle='--', alpha=0.7)
ax2.axvline(x=p_index['merchant_rank'], color='red', linestyle='--', alpha=0.7)

plt.title(f"🔍 SoukIQ Pareto Revenue Distribution: {p_index['cumulative_merchant_pct']:.1f}% of Merchants drive 80% of Profits", fontsize=14, fontweight='bold', pad=15)
fig.tight_layout()
plt.show()

In [ ]:
# 2. Extract RFM metrics from SQL
query_rfm = """
WITH MerchantRFM AS (
    SELECT 
        merchant_id,
        EXTRACT(DAY FROM ('2026-06-30 23:59:59'::TIMESTAMP - MAX(transaction_datetime))) AS recency,
        COUNT(transaction_id) AS frequency,
        SUM(amount_mad * commission_rate) AS monetary
    FROM fact_transaction
    WHERE status = 'Success'
    GROUP BY merchant_id
)
SELECT * FROM MerchantRFM;
"""

conn = get_connection()
df_rfm = pd.read_sql(query_rfm, conn)
conn.close()

# RFM Scoring using quantiles (1 to 4 scale, where 4 is best)
df_rfm['R_Score'] = pd.qcut(df_rfm['recency'], 4, labels=[4, 3, 2, 1]) # Lower recency is better
df_rfm['F_Score'] = pd.qcut(df_rfm['frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4])
df_rfm['M_Score'] = pd.qcut(df_rfm['monetary'], 4, labels=[1, 2, 3, 4])

# Calculate total RFM Score
df_rfm['RFM_Score'] = df_rfm['R_Score'].astype(int) + df_rfm['F_Score'].astype(int) + df_rfm['M_Score'].astype(int)

# Segment merchants logically
def segment_merchant(score):
    if score >= 10: return "Core VIP"
    if score >= 7: return "Loyal Active"
    if score >= 5: return "At Risk / Inactive"
    return "Churned"

df_rfm['segment'] = df_rfm['RFM_Score'].apply(segment_merchant)

# Plot RFM Segment Distribution
plt.figure(figsize=(10, 5))
colors = ['#1a365d', '#2b6cb0', '#dd6b20', '#e53e3e'] # Dark Blue, Slate Blue, Orange, Red
sns.countplot(data=df_rfm, x='segment', order=["Core VIP", "Loyal Active", "At Risk / Inactive", "Churned"], palette=colors)
plt.title("🏪 SoukIQ Hanout Segmentation based on RFM Scores", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("RFM Segment", fontsize=12)
plt.ylabel("Number of Merchants", fontsize=12)
plt.show()

In [ ]:
# 3. Extract Cohort Matrix from SQL
query_cohort = """
WITH MerchantCohorts AS (
    SELECT merchant_id, DATE_TRUNC('month', join_date) AS cohort_month FROM dim_merchant
),
MerchantActivity AS (
    SELECT DISTINCT merchant_id, DATE_TRUNC('month', transaction_datetime) AS activity_month FROM fact_transaction WHERE status = 'Success'
),
CohortTenure AS (
    SELECT 
        c.cohort_month,
        a.activity_month,
        ROUND(EXTRACT(DAY FROM (a.activity_month - c.cohort_month)) / 30.4) AS month_number,
        COUNT(DISTINCT c.merchant_id) AS active_merchants
    FROM MerchantCohorts c
    JOIN MerchantActivity a ON c.merchant_id = a.merchant_id
    GROUP BY 1, 2, 3
),
CohortSizes AS (
    SELECT cohort_month, COUNT(DISTINCT merchant_id) AS cohort_size FROM dim_merchant GROUP BY 1
)
SELECT 
    TO_CHAR(t.cohort_month, 'YYYY-MM') AS cohort_month,
    t.month_number AS tenure_month,
    s.cohort_size,
    ROUND((t.active_merchants::NUMERIC / s.cohort_size) * 100, 1) AS retention_rate_pct
FROM CohortTenure t
JOIN CohortSizes s ON t.cohort_month = s.cohort_month
WHERE t.month_number <= 5
ORDER BY t.cohort_month, t.month_number;
"""

conn = get_connection()
df_cohort = pd.read_sql(query_cohort, conn)
conn.close()

# Pivot the cohort data to create the matrix
cohort_pivot = df_cohort.pivot(index='cohort_month', columns='tenure_month', values='retention_rate_pct')

# Plotting the Cohort Heatmap (BCG style palette)
plt.figure(figsize=(10, 5))
sns.heatmap(cohort_pivot, annot=True, fmt=".1f", cmap="Blues", cbar_kws={'label': 'Retention Rate (%)'}, linewidths=.5)
plt.title("📅 MoM Merchant Cohort Retention Heatmap (H1 2026)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Months Active (Tenure Month)", fontsize=12)
plt.ylabel("Cohort Group (Join Month)", fontsize=12)
plt.show()

In [ ]:
# 4. Proving if weekend outages cause merchant churn (Timeout-Churn Correlation)
query_correlation = """
WITH OutageFrictions AS (
    -- Count failed transactions during outages for each merchant
    SELECT 
        t.merchant_id,
        COUNT(t.transaction_id) AS failed_outage_txns
    FROM fact_transaction t
    JOIN dim_outage o ON t.provider = o.provider 
        AND t.transaction_datetime BETWEEN o.start_datetime AND o.end_datetime
    WHERE t.status = 'Failed'
    GROUP BY t.merchant_id
),
MerchantRecency AS (
    -- Days inactive relative to the end of H1 2026
    SELECT 
        merchant_id,
        EXTRACT(DAY FROM ('2026-06-30 23:59:59'::TIMESTAMP - MAX(transaction_datetime))) AS days_inactive
    FROM fact_transaction
    WHERE status = 'Success'
    GROUP BY merchant_id
)
SELECT 
    r.merchant_id,
    COALESCE(f.failed_outage_txns, 0) AS failed_outage_txns,
    r.days_inactive
FROM MerchantRecency r
LEFT JOIN OutageFrictions f ON r.merchant_id = f.merchant_id;
"""

conn = get_connection()
df_corr = pd.read_sql(query_correlation, conn)
conn.close()

# Calculate Pearson Correlation
correlation_coefficient = df_corr['failed_outage_txns'].corr(df_corr['days_inactive'])

# Visual representation of Outage Failures vs. Inactivity
plt.figure(figsize=(10, 5))
sns.boxplot(data=df_corr, x='failed_outage_txns', y='days_inactive', palette='Reds')
plt.title(f"🔌 Tech-Friction vs. Merchant Churn | Pearson Correlation: {correlation_coefficient:.3f}", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Number of API Failures Experienced During Weekend Outages", fontsize=12)
plt.ylabel("Days of Inactivity (At End of H1 2026)", fontsize=12)
plt.show()

# Print business summary
print("="*60)
print("                   BUSINESS CORRELATION ANALYSIS             ")
print("="*60)
print(f"👉 Pearson Correlation: {correlation_coefficient:.3f}")
print("👉 Root Cause Insight: Merchants experiencing more API timeouts during outages")
print("   exhibit significantly longer periods of inactivity, proving technical friction")
print("   is a primary driver of merchant silent churn.")